
# TDE Radio Afterglow: Mildly Relativistic Outflow in ISM

Many TDEs launch a **mildly relativistic outflow** — either a disk wind, a
reprocessing shell, or an off-axis jet — that expands into the surrounding
interstellar medium (ISM). This outflow drives a forward shock that accelerates
electrons and amplifies the magnetic field, producing **delayed radio brightening**
months to years after the optical TDE peak.

In this example we model an outflow launched at $v_{\rm init} = 0.3\,c$
with total kinetic energy $E_{\rm ej} = 10^{50}$ erg. The ejecta mass
follows from the kinetic energy and launch velocity:

\begin{align}M_{\rm ej} = \frac{2 E_{\rm ej}}{v_{\rm init}^2} \approx 1.2 \times 10^{-3}\,M_\odot.\end{align}

The shock dynamics are computed using
:class:`~triceratops.dynamics.shocks.numerical.PressureDrivenThinShellShockEngine`
with a uniform-density ISM of $n_{\rm ISM} = 10\,\text{cm}^{-3}$, placing the
Sedov-Taylor transition at roughly 500 days. The synchrotron emission is then
computed using the standard forward-closure machinery.

.. hint::

    TDE outflow parameters $E_{\rm ej}$ and $n_{\rm ISM}$ are
    the main unknowns — together with the microphysical parameters
    $\epsilon_e$ and $\epsilon_B$. This example shows how to
    predict the radio light curves given these parameters.


## Setup



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy import constants as const
from astropy import units as u

from triceratops.dynamics.shocks.numerical import PressureDrivenThinShellShockEngine
from triceratops.dynamics.shocks.utils import (
    get_bpl_ejecta_kernel,
    get_uniform_csm_density_func,
    make_homologous_stationary_sources,
)
from triceratops.radiation.synchrotron import PowerLaw_Cooling_SSA_SynchrotronSED
from triceratops.radiation.synchrotron.cooling import SynchrotronRadiativeCoolingEngine
from triceratops.utils.plot_utils import set_plot_style

set_plot_style()

## Physical Parameters

We set the launch velocity to $0.3\,c$ and kinetic energy to
$10^{50}$ erg, then derive the ejecta mass from the kinetic energy
relation $E_{\rm ej} = \tfrac{1}{2} M_{\rm ej} v^2$.



In [ ]:
# ------ Outflow parameters ------
E_ej = 1.0e50 * u.erg
v_init = (0.3 * const.c).to(u.cm / u.s)
M_ej = (2.0 * E_ej / v_init**2).to(u.M_sun)

n_ISM = 10.0 / u.cm**3
mu = 0.61  # Mean molecular weight (fully ionized)
rho_ISM = (mu * const.m_p * n_ISM).to(u.g / u.cm**3)

# ------ Microphysical parameters ------
epsilon_e = 0.1
epsilon_B = 0.1
p = 3.0
gamma_min = 1.0
gamma_max = 1e8
f_V = 1.0
f_A = 1.0

# ------ Observing setup ------
D_L = 200.0 * u.Mpc
obs_frequencies = u.Quantity([1.4, 5.0, 15.0], u.GHz)
times = np.geomspace(20.0, 3000.0, 100) * u.day

print(f"Launch velocity : {v_init.to(u.km / u.s):.0f}  ({(v_init / const.c).decompose():.2f} c)")
print(f"Outflow energy  : {E_ej:.1e}")
print(f"Outflow mass    : {M_ej:.4f}")
print(f"ISM density     : {n_ISM:.0f}")
print(f"Distance        : {D_L}")

## Shock Dynamics in Uniform ISM

For a blast wave expanding into a uniform ISM the shock decelerates as it
sweeps up mass. Initially the ejecta are in **free expansion**
($R \propto t$), then transition to the **Sedov-Taylor** phase
($R \propto t^{2/5}$) once the swept-up mass equals the ejecta mass.
With these parameters the transition occurs at roughly 500 days.

We use :func:`~triceratops.dynamics.shocks.utils.get_uniform_csm_density_func`
and :func:`~triceratops.dynamics.shocks.utils.get_bpl_ejecta_kernel` to build
the upstream source callables, then combine them with
:func:`~triceratops.dynamics.shocks.utils.make_homologous_stationary_sources`.



In [ ]:
rho_csm = get_uniform_csm_density_func(rho_ISM)
G_ej = get_bpl_ejecta_kernel(E_ej, M_ej, n=10, delta=0)
rho_1, u_1, rho_4, u_4 = make_homologous_stationary_sources(G_ej=G_ej, rho_csm=rho_csm)

t_init = 10.0 * u.day
R_init = (v_init * t_init).to(u.cm)

shock_engine = PressureDrivenThinShellShockEngine()
state = shock_engine.compute_shock_properties(
    times,
    rho_1=rho_1,
    rho_4=rho_4,
    u_1=u_1,
    u_4=u_4,
    R_0=R_init,
    v_0=v_init,
    t_0=t_init,
    M_0=1e-8 * u.M_sun,
)

r_sh = state.radius.to(u.cm)
v_sh = state.velocity.to(u.cm / u.s)
M_sw = state.mass.to(u.M_sun)

## Sedov-Taylor Transition

We estimate the Sedov-Taylor transition time as the moment when the
swept-up ISM mass equals the ejecta mass.



In [ ]:
M_sw_over_Mej = (M_sw / M_ej).decompose()
idx_ST = np.argmin(np.abs(M_sw_over_Mej.value - 1.0))
t_ST = times[idx_ST]
R_ST = r_sh[idx_ST]

print(f"\nSedov-Taylor transition:")
print(f"  t_ST = {t_ST:.0f}")
print(f"  R_ST = {R_ST:.3e}")
print(f"  v_ST = {v_sh[idx_ST].to(u.km / u.s):.0f}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True)

axes[0].loglog(times.to_value(u.day), r_sh.to_value(u.cm), color="C0", lw=2)
axes[0].axvline(t_ST.to_value(u.day), ls="--", color="gray", label=f"Sedov-Taylor ({t_ST.to_value(u.day):.0f} d)")
axes[0].set_ylabel(r"$R_{\rm sh}$ [cm]")
axes[0].set_title("Shock Radius")
axes[0].legend(fontsize=8)
axes[0].grid(True, which="both", ls="--", alpha=0.3)

axes[1].loglog(times.to_value(u.day), (v_sh / const.c.cgs).decompose().value, color="C1", lw=2)
axes[1].axvline(t_ST.to_value(u.day), ls="--", color="gray")
axes[1].set_ylabel(r"$v_{\rm sh}/c$")
axes[1].set_title("Shock Velocity")
axes[1].set_xlabel("Time [days]")
axes[1].grid(True, which="both", ls="--", alpha=0.3)

axes[2].loglog(times.to_value(u.day), M_sw.to_value(u.M_sun), color="C2", lw=2)
axes[2].axhline(M_ej.to_value(u.M_sun), ls="--", color="C3", label=r"$M_{\rm ej}$")
axes[2].axvline(t_ST.to_value(u.day), ls="--", color="gray")
axes[2].set_ylabel(r"$M_{\rm swept}$ [$M_\odot$]")
axes[2].set_title("Swept-up Mass")
axes[2].legend(fontsize=8)
axes[2].grid(True, which="both", ls="--", alpha=0.3)

plt.suptitle("TDE Outflow Shock Dynamics in Uniform ISM", fontsize=12)
plt.tight_layout()
plt.show()

## Post-Shock Magnetic Field

The magnetic field is estimated from equipartition with the post-shock thermal
energy density. The
:class:`~triceratops.dynamics.shocks.numerical.ThinShellShockState` carries
the immediate post-shock pressure at the forward shock from the
Rankine--Hugoniot jump conditions, so the thermal energy density is

\begin{align}e_{\rm th} = \frac{p_s}{\gamma - 1},\end{align}

and the equipartition magnetic field is

\begin{align}B = \sqrt{8\pi\,\epsilon_B\,e_{\rm th}}.\end{align}



In [ ]:
gamma_sh = 5.0 / 3.0
e_th = state.post_shock_pressure / (gamma_sh - 1.0)
B = np.sqrt(8.0 * np.pi * epsilon_B * e_th.to_value(u.dyn / u.cm**2)) * u.G

cooling_engine = SynchrotronRadiativeCoolingEngine()
gamma_c = cooling_engine.compute_cooling_gamma(B=B, t=times)

print(f"\nB-field range : {B.min():.4f} - {B.max():.4f}")
print(f"gamma_c range : {gamma_c.min():.1e} - {gamma_c.max():.1e}")

## Synchrotron Radio Light Curves

We compute the synchrotron SED at each epoch using the
:class:`~triceratops.radiation.synchrotron.SEDs.PowerLaw_Cooling_SSA_SynchrotronSED`
forward closure and evaluate the predicted flux at 1.4, 5, and 15 GHz.



In [ ]:
sed = PowerLaw_Cooling_SSA_SynchrotronSED()

lcs = {freq: np.zeros(len(times)) for freq in obs_frequencies.to_value(u.GHz)}

for i, (r, B_i, gc) in enumerate(zip(r_sh, B, gamma_c)):
    sed_params = sed.from_physics_to_params(
        B=B_i,
        R=r,
        gamma_min=gamma_min,
        gamma_max=gamma_max,
        p=p,
        epsilon_E=epsilon_e,
        epsilon_B=epsilon_B,
        luminosity_distance=D_L,
        f_V=f_V,
        f_A=f_A,
        gamma_c=float(gc),
        pitch_average=True,
    )
    for freq in obs_frequencies:
        flux = sed.sed(
            nu=freq.to(u.Hz),
            nu_m=sed_params["nu_m"],
            nu_c=sed_params["nu_c"],
            F_norm=sed_params["F_norm"],
            nu_max=sed_params["nu_max"],
            nu_ac=sed_params["nu_a"],
            omega=sed_params["omega"],
            p=p,
        ).to(u.mJy)
        lcs[freq.to_value(u.GHz)][i] = flux.value

## Plot Radio Light Curves

The characteristic signature of a TDE radio afterglow: the flux density
rises as the shock sweeps up ISM and the synchrotron-emitting volume grows,
peaks near the Sedov-Taylor transition, then declines as the shock
decelerates. Lower frequencies peak later because the SSA absorption
frequency $\nu_a$ falls with time.



In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for freq, color in zip(obs_frequencies, ["C0", "C1", "C2"]):
    freq_ghz = freq.to_value(u.GHz)
    ax.loglog(times.to_value(u.day), lcs[freq_ghz], lw=2.5, color=color, label=f"{freq_ghz:.1f} GHz")

ax.axvline(t_ST.to_value(u.day), ls="--", color="gray", lw=1.5, label=f"Sedov-Taylor ({t_ST.to_value(u.day):.0f} d)")
ax.set_xlabel("Time post-disruption [days]")
ax.set_ylabel("Flux Density [mJy]")
ax.set_title(
    rf"TDE Radio Afterglow: $E_{{\rm ej}} = 10^{{50}}$ erg, "
    rf"$v_{{\rm init}} = 0.3\,c$, $n_{{\rm ISM}} = 10\,\mathrm{{cm}}^{{-3}}$, $D = {D_L.to_value(u.Mpc):.0f}$ Mpc"
)
ax.legend()
ax.grid(True, which="both", ls="--", alpha=0.3)
plt.tight_layout()
plt.show()

## Energy Scaling: Effect of Outflow Energy

The peak flux and peak time scale with outflow energy. We sweep three
values of $E_{\rm ej}$ while keeping the launch velocity fixed at
$0.3\,c$, so the ejecta mass scales as $M_{\rm ej} = 2 E_{\rm ej} / v^2$
for each case.



In [ ]:
E_ej_values = [1e49, 1e50, 1e51] * u.erg
E_labels = [r"$10^{49}$ erg", r"$10^{50}$ erg", r"$10^{51}$ erg"]

fig, ax = plt.subplots(figsize=(9, 5))

for E, label, color in zip(E_ej_values, E_labels, ["C0", "C1", "C2"]):
    M_ej_E = (2.0 * E / v_init**2).to(u.M_sun)

    G_ej_E = get_bpl_ejecta_kernel(E, M_ej_E, n=10, delta=0)
    rho_1_E, u_1_E, rho_4_E, u_4_E = make_homologous_stationary_sources(G_ej=G_ej_E, rho_csm=rho_csm)

    state_E = shock_engine.compute_shock_properties(
        times,
        rho_1=rho_1_E,
        rho_4=rho_4_E,
        u_1=u_1_E,
        u_4=u_4_E,
        R_0=R_init,
        v_0=v_init,
        t_0=t_init,
        M_0=1e-8 * u.M_sun,
    )

    e_th_E = state_E.post_shock_pressure / (gamma_sh - 1.0)
    B_E = np.sqrt(8.0 * np.pi * epsilon_B * e_th_E.to_value(u.dyn / u.cm**2)) * u.G
    gc_E = cooling_engine.compute_cooling_gamma(B=B_E, t=times)

    lc_5ghz = np.zeros(len(times))
    freq_eval = 5.0 * u.GHz

    for i, (r, B_i, gc) in enumerate(zip(state_E.radius.to(u.cm), B_E, gc_E)):
        sp = sed.from_physics_to_params(
            B=B_i,
            R=r,
            gamma_min=gamma_min,
            gamma_max=gamma_max,
            p=p,
            epsilon_E=epsilon_e,
            epsilon_B=epsilon_B,
            luminosity_distance=D_L,
            f_V=f_V,
            f_A=f_A,
            gamma_c=float(gc),
            pitch_average=True,
        )
        lc_5ghz[i] = (
            sed.sed(
                nu=freq_eval.to(u.Hz),
                nu_m=sp["nu_m"],
                nu_c=sp["nu_c"],
                F_norm=sp["F_norm"],
                nu_max=sp["nu_max"],
                nu_ac=sp["nu_a"],
                omega=sp["omega"],
                p=p,
            )
            .to(u.mJy)
            .value
        )

    ax.loglog(times.to_value(u.day), lc_5ghz, lw=2, color=color, label=label)

ax.set_xlabel("Time post-disruption [days]")
ax.set_ylabel("Flux Density at 5 GHz [mJy]")
ax.set_title(r"Radio Afterglow Sensitivity to Outflow Energy ($v_{\rm init} = 0.3\,c$, 5 GHz)")
ax.legend()
ax.grid(True, which="both", ls="--", alpha=0.3)
plt.tight_layout()
plt.show()

## Interpretation

The TDE radio afterglow has several characteristic observational signatures:

- **Delayed peak**: the radio emission peaks months to years after the optical
  TDE, depending on $E_{\rm ej}$ and $n_{\rm ISM}$.
- **Frequency ordering**: lower frequencies peak later, reflecting the
  declining SSA frequency as the shock expands.
- **Steep rise**: during the free-expansion phase the flux rises steeply
  because the synchrotron-emitting volume grows as $R^3 \propto t^3$.
- **Post-peak decline**: after the Sedov-Taylor transition the shock decelerates
  and the flux falls as the synchrotron emission decreases.
- **Energy scaling**: higher-energy outflows produce brighter and later-peaking
  emission, since a larger ejecta mass takes longer to sweep up an equal
  ISM mass.

Comparing model light curves to observed TDE radio data can constrain
the total outflow kinetic energy, the ISM density at the black hole's sphere
of influence, and (via the microphysical parameters) the efficiency of
particle acceleration at the shock.

